# Shared UMAP Space: Do Title and Track Embeddings Land in the Same Cluster?

This notebook takes a different approach from the separate-UMAP analysis. Instead of running UMAP independently on title and track embeddings and comparing the resulting clusterings with ARI, we **combine both embedding types into a single matrix** and run UMAP once on everything together.

Each playlist contributes **two rows** to the combined matrix:
- One row for its title embedding
- One row for its tracks embedding

After UMAP + KMeans, we check how often a playlist's title row and its track row ended up in the **same cluster**.

**Research question:** In a shared embedding space, do a playlist's title and its tracks land near each other — or do they occupy completely different regions?

| Approach | What it measures |
|---|---|
| Separate UMAPs + ARI (previous notebooks) | Do the two modalities produce *similar partition structures* across all playlists? |
| **This notebook** | Do the title and tracks of the *same playlist* land near each other in shared space? |

**Pipeline:**
1. Load & sample the data
2. Normalize both embedding matrices
3. Stack into one combined matrix (200k rows)
4. Run UMAP once on the combined matrix
5. Cluster with KMeans
6. Check same-cluster rate per playlist
7. Visualize — color by modality to see if titles and tracks mix or separate
8. Visualize — color by cluster
9. Investigate specific playlists where title and track landed in the same vs different clusters

## Step 1 — Mount Google Drive and Install Packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install umap-learn plotly pandas scikit-learn kaleido -q

## Step 2 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import umap
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from datetime import datetime

## Step 2b — Fix Pandas Version (if needed)

The pickle file was saved with a newer version of pandas than Colab's default. Run this cell **once**, then **restart the runtime** and re-run from the top. Skip if pandas is already up to date.

In [ ]:
# Uncomment and run once if you get a pandas version error when loading the pickle
# import subprocess, sys, os
# subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pandas", "-q"])
# os.kill(os.getpid(), 9)  # Restart runtime — re-run all cells after this

## Step 3 — Load the Pickle File

In [ ]:
import pickle

path = "/content/drive/MyDrive/Colab Notebooks/embeddings.pkl"
with open(path, "rb") as f:
    df = pickle.load(f)

print("Full DataFrame shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

## Step 4 — Sample the Data

In [ ]:
SAMPLE_SIZE = 100_000
RANDOM_STATE = 42
N_CLUSTERS = 50

df_sample = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
print("Sampled shape:", df_sample.shape)

## Step 5 — Extract, Normalize, and Stack into a Combined Matrix

We stack title and track embeddings into a single matrix of shape `(200000, 384)`.
The first 100k rows are title embeddings; the last 100k rows are track embeddings.
This pairing is preserved by index — row `i` in titles corresponds to row `i + n` in tracks,
and both refer to `df_sample.iloc[i]`.

In [ ]:
n = len(df_sample)

# Extract
title_embeddings = np.vstack(df_sample['title_embedding'].values)
track_embeddings = np.vstack(df_sample['tracks_embedding'].values)

# Normalize
title_embeddings_norm = normalize(title_embeddings)
track_embeddings_norm = normalize(track_embeddings)

# Stack: titles first, then tracks
combined = np.vstack([title_embeddings_norm, track_embeddings_norm])
print("Combined matrix shape:", combined.shape)
print(f"Rows 0 to {n-1}: title embeddings")
print(f"Rows {n} to {2*n-1}: track embeddings")

In [ ]:
# #alternate code for step 5 if we do NOT want to normalize the embeddings
# n = len(df_sample)

# # Extract
# title_embeddings = np.vstack(df_sample['title_embedding'].values)
# track_embeddings = np.vstack(df_sample['tracks_embedding'].values)

# # Stack raw (no normalization)
# combined = np.vstack([title_embeddings, track_embeddings])
# print("Combined matrix shape:", combined.shape)
# print(f"Rows 0 to {n-1}: title embeddings")
# print(f"Rows {n} to {2*n-1}: track embeddings")

## Step 6 — Run UMAP on the Combined Matrix

A single UMAP fit on all 200k rows. Both title and track embeddings are projected into the **same 2D coordinate space**, so their positions are directly comparable.

This is the key difference from the previous notebooks: here, a title point and its matching track point can be close or far apart in a meaningful way.

In [ ]:
umap_combined = umap.UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.1,
    metric="cosine",
    random_state=RANDOM_STATE
)

combined_2d = umap_combined.fit_transform(combined)
print("Combined UMAP output shape:", combined_2d.shape)

# Save to Drive so you don't have to rerun
np.save("/content/drive/MyDrive/Colab Notebooks/combined_2d.npy", combined_2d)
print("Saved combined_2d.npy")

## Step 6b — Reload Saved Array (use this if restarting)

Skip Step 6 and run this instead if the UMAP array is already saved to Drive.

In [ ]:
# Uncomment to reload without refitting UMAP
# combined_2d = np.load("/content/drive/MyDrive/Colab Notebooks/combined_2d.npy")
# print("Loaded combined_2d.npy, shape:", combined_2d.shape)

## Step 7 — Split Projections Back Out

Now that we have 2D coordinates for all 200k rows, we split them back into title coordinates and track coordinates. Row `i` in `title_2d` and row `i` in `track_2d` correspond to the same playlist.

In [ ]:
title_2d = combined_2d[:n]   # first 100k rows
track_2d = combined_2d[n:]   # last 100k rows

print("title_2d shape:", title_2d.shape)
print("track_2d shape:", track_2d.shape)

## Step 8 — Cluster the Combined Projection

We run KMeans **once** on all 200k points together. This means cluster boundaries are shared — a title and its matching track are either in the same cluster or not, in a directly meaningful sense.

In [ ]:
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init='auto')
all_labels = kmeans.fit_predict(combined_2d)

# Split labels back out
title_labels = all_labels[:n]
track_labels = all_labels[n:]

print("Title label distribution:")
print(pd.Series(title_labels).value_counts().sort_index())
print("\nTrack label distribution:")
print(pd.Series(track_labels).value_counts().sort_index())

## Step 9 — Same-Cluster Rate

For each playlist, we check whether its title and its tracks landed in the same KMeans cluster.

- **High same-cluster rate** → title and track embeddings are geometrically close in the shared space; the two modalities agree on what each playlist is "about"
- **Low same-cluster rate** → title and track embeddings occupy different regions; knowing a playlist's title tells you little about its musical content

We also compute ARI for comparison with the previous notebooks — though note it measures something slightly different here since clustering was done jointly.

In [ ]:
same_cluster = (title_labels == track_labels)
same_cluster_rate = same_cluster.mean()

print(f"Same cluster: {same_cluster.sum():,} / {n:,} playlists ({same_cluster_rate:.1%})")
print()

# Baseline: if cluster assignment were random, how often would they match by chance?
# With k=50 clusters and ~uniform distribution, chance rate ≈ 1/50 = 2%
chance_rate = 1 / N_CLUSTERS
print(f"Chance baseline (1/k): {chance_rate:.1%}")
print(f"Lift over chance: {same_cluster_rate / chance_rate:.1f}x")
print()

# ARI for comparison
ari = adjusted_rand_score(title_labels, track_labels)
print(f"ARI (for reference): {ari:.4f}")

## Step 10 — Build the Plot DataFrame

Assemble a single DataFrame with coordinates, cluster labels, modality, and playlist metadata for all 200k points.

In [ ]:
plot_df = pd.DataFrame({
    "x": combined_2d[:, 0],
    "y": combined_2d[:, 1],
    "cluster": all_labels.astype(str),
    "modality": ["title"] * n + ["track"] * n,
    "playlist_title": list(df_sample['playlist_title'].values) * 2,
    "playlist_index": list(range(n)) * 2
})

# Add same-cluster flag (only meaningful for title rows, but we'll use it for filtering)
same_cluster_col = list(same_cluster.astype(str)) + list(same_cluster.astype(str))
plot_df["same_cluster"] = same_cluster_col

print("Plot DataFrame shape:", plot_df.shape)
print(plot_df.head())

## Step 11 — Visualize: Color by Modality

This is the **most important diagnostic plot**. Color each point by whether it's a title or a track embedding.

- **If titles and tracks form two separate blobs** → the two modalities live in very different regions of embedding space. UMAP is separating them before cluster structure even comes into play. Same-cluster rate will be near the chance baseline.
- **If titles and tracks are interleaved** → the two modalities overlap in embedding space. Title and track embeddings for the same playlist may genuinely land near each other.

In [ ]:
fig_modality = px.scatter(
    plot_df,
    x="x", y="y",
    color="modality",
    color_discrete_map={"title": "#636EFA", "track": "#EF553B"},
    hover_data=["playlist_title", "cluster"],
    title="Combined UMAP — Colored by Modality (Title vs Track)",
    opacity=0.3,
    width=900, height=700
)
fig_modality.show()

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
fig_modality.write_image(f"/content/drive/MyDrive/Colab Notebooks/umap_combined_modality_{timestamp}.png")
print("Saved modality plot.")

## Step 12 — Visualize: Color by Cluster

Same plot but colored by KMeans cluster. This shows whether the cluster structure cuts across both modalities or mostly within one.

In [ ]:
fig_cluster = px.scatter(
    plot_df,
    x="x", y="y",
    color="cluster",
    symbol="modality",
    hover_data=["playlist_title", "modality"],
    title="Combined UMAP — Colored by Cluster, Symbol by Modality",
    opacity=0.3,
    width=900, height=700
)
fig_cluster.show()

fig_cluster.write_image(f"/content/drive/MyDrive/Colab Notebooks/umap_combined_cluster_{timestamp}.png")
print("Saved cluster plot.")

## Step 13 — Visualize: Same-Cluster Pairs

Plot only the title points, colored by whether their matching track ended up in the same cluster.

This gives a spatial view of *where* title-track agreement happens across the UMAP layout.

In [ ]:
# Title points only
title_plot_df = pd.DataFrame({
    "x": title_2d[:, 0],
    "y": title_2d[:, 1],
    "cluster": title_labels.astype(str),
    "same_cluster": same_cluster,
    "playlist_title": df_sample['playlist_title'].values
})

fig_same = px.scatter(
    title_plot_df,
    x="x", y="y",
    color="same_cluster",
    color_discrete_map={True: "#00CC96", False: "#EF553B"},
    hover_data=["playlist_title", "cluster"],
    title="Title Points — Green = Title & Track in Same Cluster, Red = Different Cluster",
    opacity=0.4,
    width=900, height=700
)
fig_same.show()

fig_same.write_image(f"/content/drive/MyDrive/Colab Notebooks/umap_combined_same_cluster_{timestamp}.png")
print("Saved same-cluster plot.")

## Step 14 — Inspect Specific Playlists

Look at example playlists where title and track **did** land in the same cluster vs where they **didn't**.

This is a qualitative sanity check — do the same-cluster playlists make intuitive sense? Are the different-cluster ones surprising or explainable?

In [ ]:
# Add results back to df_sample for easy inspection
df_sample['title_cluster'] = title_labels
df_sample['track_cluster'] = track_labels
df_sample['same_cluster'] = same_cluster
df_sample['title_x'] = title_2d[:, 0]
df_sample['title_y'] = title_2d[:, 1]
df_sample['track_x'] = track_2d[:, 0]
df_sample['track_y'] = track_2d[:, 1]

# Distance between title and track point in UMAP space
df_sample['umap_distance'] = np.sqrt(
    (df_sample['title_x'] - df_sample['track_x'])**2 +
    (df_sample['title_y'] - df_sample['track_y'])**2
)

print("=" * 60)
print("PLAYLISTS WHERE TITLE AND TRACK LANDED IN THE SAME CLUSTER")
print("(closest pairs — most agreement)")
print("=" * 60)
same = df_sample[df_sample['same_cluster']].nsmallest(10, 'umap_distance')
for _, row in same.iterrows():
    print(f"  Title: '{row['playlist_title']}'")
    print(f"  Cluster: {row['title_cluster']} | UMAP distance: {row['umap_distance']:.3f}")
    print()

print("=" * 60)
print("PLAYLISTS WHERE TITLE AND TRACK LANDED IN DIFFERENT CLUSTERS")
print("(furthest pairs — most disagreement)")
print("=" * 60)
diff = df_sample[~df_sample['same_cluster']].nlargest(10, 'umap_distance')
for _, row in diff.iterrows():
    print(f"  Title: '{row['playlist_title']}'")
    print(f"  Title cluster: {row['title_cluster']} | Track cluster: {row['track_cluster']} | UMAP distance: {row['umap_distance']:.3f}")
    print()

## Step 15 — Per-Cluster Same-Cluster Rate

Break down the same-cluster rate by title cluster. Some clusters may show much stronger title-track agreement than others — which could reveal which *types* of playlists have titles that predict their musical content well.

In [ ]:
cluster_stats = df_sample.groupby('title_cluster').agg(
    count=('same_cluster', 'count'),
    same_cluster_rate=('same_cluster', 'mean')
).reset_index().sort_values('same_cluster_rate', ascending=False)

print("Same-cluster rate by title cluster (top 10):")
print(cluster_stats.head(10).to_string(index=False))
print()
print("Same-cluster rate by title cluster (bottom 10):")
print(cluster_stats.tail(10).to_string(index=False))

# Bar chart
fig_bar = px.bar(
    cluster_stats.sort_values('title_cluster'),
    x='title_cluster', y='same_cluster_rate',
    title='Same-Cluster Rate by Title Cluster',
    labels={'title_cluster': 'Title Cluster', 'same_cluster_rate': 'Same-Cluster Rate'},
    width=1000, height=450
)
fig_bar.add_hline(y=same_cluster_rate, line_dash="dash",
                  annotation_text=f"Overall avg: {same_cluster_rate:.1%}")
fig_bar.show()

fig_bar.write_image(f"/content/drive/MyDrive/Colab Notebooks/same_cluster_by_cluster_{timestamp}.png")
print("Saved per-cluster bar chart.")

## Step 16 — Summary

In [ ]:
print("=" * 55)
print("RESULTS SUMMARY")
print("=" * 55)
print(f"Sample size:              {SAMPLE_SIZE:,}")
print(f"KMeans clusters:          {N_CLUSTERS}")
print(f"Total points in UMAP:     {SAMPLE_SIZE*2:,} (titles + tracks)")
print()
print(f"Same-cluster rate:        {same_cluster_rate:.1%}")
print(f"Chance baseline (1/k):    {chance_rate:.1%}")
print(f"Lift over chance:         {same_cluster_rate / chance_rate:.1f}x")
print()
print(f"ARI (title vs track):     {ari:.4f}")
print()
print(f"Median UMAP distance (same cluster):  {df_sample[df_sample['same_cluster']]['umap_distance'].median():.3f}")
print(f"Median UMAP distance (diff cluster):  {df_sample[~df_sample['same_cluster']]['umap_distance'].median():.3f}")
print()
print("Interpretation:")
if same_cluster_rate > 0.5:
    print("  Strong alignment — titles and tracks land in the same cluster")
    print("  for the majority of playlists. The two modalities agree.")
elif same_cluster_rate > 0.1:
    print("  Moderate alignment — titles and tracks co-cluster meaningfully")
    print("  above chance, but agreement is partial.")
else:
    print("  Weak alignment — title and track embeddings occupy largely")
    print("  separate regions of the shared embedding space.")
    print("  Check the modality plot (Step 11) to see if they form two blobs.")